# gpuless runtime builder

Builds the `gpuless-runtime` dataset **on Kaggle itself**, so the Python packages match the image the runner uses.

Before you run it, open the notebook settings on the right:

- **Accelerator:** None (this notebook needs no GPU and spends none of your GPU hours)
- **Internet:** On

Then click **Save Version > Save & Run All (Commit)**. When the version has finished, create a dataset from its output and call it `gpuless-runtime`.

The output has exactly the layout the runner expects:

```
ComfyUI/            the checkout
site-packages/      only what the Kaggle image does not already have
bin/cloudflared     the tunnel client
```

In [ ]:
import json, os, re, shutil, subprocess, sys, urllib.request

W = "/kaggle/working"
os.chdir(W)

def run(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

# start clean if the notebook is run twice
for name in ("ComfyUI", "site-packages", "bin"):
    shutil.rmtree(name, ignore_errors=True)

run("git clone --depth 1 https://github.com/comfyanonymous/ComfyUI ComfyUI")
shutil.rmtree("ComfyUI/.git", ignore_errors=True)

In [ ]:
# Ask pip what it would have to install on top of the Kaggle image, without installing anything.
# torch and its CUDA libraries are already in the image and are several gigabytes, so they are never copied.
SKIP = re.compile(r"^(torch|torchvision|torchaudio|triton|nvidia-.*)$")

reqs = []
for line in open("ComfyUI/requirements.txt"):
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    name = re.split(r"[<>=!~\[ ;]", line)[0].lower()
    if not SKIP.match(name):
        reqs.append(line)
open("/tmp/requirements.txt", "w").write("\n".join(reqs) + "\n")

run(f"{sys.executable} -m pip install --quiet --dry-run --report /tmp/report.json -r /tmp/requirements.txt")
report = json.load(open("/tmp/report.json"))
missing = []
for item in report.get("install", []):
    name, version = item["metadata"]["name"], item["metadata"]["version"]
    if not SKIP.match(name.lower()):
        missing.append(f"{name}=={version}")
print(len(missing), "packages to add:", ", ".join(missing))

if missing:
    open("/tmp/missing.txt", "w").write("\n".join(missing) + "\n")
    run(f"{sys.executable} -m pip install --quiet --no-cache-dir --no-deps --target {W}/site-packages -r /tmp/missing.txt")
else:
    os.makedirs("site-packages", exist_ok=True)

In [ ]:
os.makedirs("bin", exist_ok=True)
urllib.request.urlretrieve(
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    "bin/cloudflared")
os.chmod("bin/cloudflared", 0o755)
run("bin/cloudflared --version")

In [ ]:
# Self-test: start ComfyUI the way the runner does, on the CPU, and let it exit right after loading.
env = dict(os.environ, PYTHONPATH=f"{W}/site-packages")
subprocess.run([sys.executable, "main.py", "--cpu", "--quick-test-for-ci"], cwd="ComfyUI", env=env, check=True)

run("du -sh ComfyUI site-packages bin")
print("\nDone. Create a dataset from this version's output and call it gpuless-runtime.")